<a href="https://colab.research.google.com/github/ferminhung/nubox_challenge/blob/main/nubox_challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import auth
auth.authenticate_user()

In [2]:
def init_ev():
  from google.colab import userdata

  openaq_api_key = userdata.get('openaq_api_key')
  return openaq_api_key

In [3]:
class OpenAQ:
  import requests

  def __init__(self,
               openaq_api_key: str
  ) -> None:
      self.openaq_api_key = openaq_api_key

  def get_locations(self, id_location):
    url = f'https://api.openaq.org/v3/locations/{id_location}'
    headers = {
      'accept': 'application/json',
      'X-API-Key': self.openaq_api_key
    }

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
      data = response.json()
      return data['results']
    else:
      print(f"Error: {response.status_code}")
      return response.text

  def get_countries(self):
    url = 'https://api.openaq.org/v3/countries'
    headers = {
      'accept': 'application/json',
      'X-API-Key': self.openaq_api_key
    }

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
      data = response.json()
      return data['results']
    else:
      print(f"Error: {response.status_code}")
      return response.text

  def get_localities(self):
    url = 'https://api.openaq.org/v3/locations?limit=100&page=1&order_by=id&sort_order=asc&countries_id=3'
    headers = {
      'accept': 'application/json',
      'X-API-Key': self.openaq_api_key
    }

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
      data = response.json()
      return data['results']
    else:
      print(f"Error: {response.status_code}")
      return response.text

  def get_sensor(self, sensor_id):
    url = f'https://api.openaq.org/v3/sensors/{sensor_id}/measurements/daily?datetime_to=2025-10-19&datetime_from=2025-10-13&limit=100&page=1'
    headers = {
      'accept': 'application/json',
      'X-API-Key': self.openaq_api_key
    }

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
      data = response.json()
      return data['results']
    else:
      print(f"Error: {response.status_code}")
      return response.text

In [4]:
from ctypes import Array
class JsonFileAQ:
  import json
  import os
  from datetime import datetime



  def __init__(self, file_name: str, data_str: str
  ) -> None:
      self.file_name = file_name
      self.data_str = data_str
      self.local_file_name = ""
      self.batch_date = ""

  def create_local_json_file(self) -> Array[str,str,str]:
    now = datetime.now()
    self.batch_date = now.strftime("%Y-%m-%d")
    self.local_file_name = f"{self.file_name}_{batch_date}.json"
    # Save the dictionary to a JSON file
    with open(self.local_file_name, 'w') as f:
        json.dump(self.data_str, f)

    return [f"'{self.local_file_name}' created successfully.", self.batch_date, self.local_file_name]

  def get_normalized_dataframe(self, bucket_name):
    downloaded_file_name = f"/content/{self.local_file_name}"

    source_blob_name = f"raw_data_openaq/{self.file_name}/{batch_date}/{self.local_file_name}" # Define the source path in the bucket

    uploader = GoogleStorageUploader(bucket_name)
    uploader.download_file(source_blob_name, downloaded_file_name)

    with open(downloaded_file_name, 'r') as f:
        data = json.load(f)

    df = pd.json_normalize(data,sep="_")
    return df

In [6]:
from google.cloud import storage

class GoogleStorageUploader:
    def __init__(self, bucket_name):
        self.bucket_name = bucket_name
        self.storage_client = storage.Client()
        self.bucket = self.storage_client.bucket(self.bucket_name)

    def upload_file(self, source_file_name, destination_blob_name):
        """Uploads a file to the bucket."""
        blob = self.bucket.blob(destination_blob_name)
        blob.upload_from_filename(source_file_name)
        print(f"File {source_file_name} uploaded to {destination_blob_name} in bucket {self.bucket_name}.")

    def download_file(self, source_blob_name, destination_file_name):
        """Downloads a file from the bucket."""
        blob = self.bucket.blob(source_blob_name)
        blob.download_to_filename(destination_file_name)
        print(f"Blob {source_blob_name} downloaded to {destination_file_name}.")

# Example usage:
# uploader = GoogleStorageUploader("your-bucket-name")
# uploader.upload_file("local/path/to/your/file.txt", "destination/path/in/bucket/file.txt")
# uploader.download_file("destination/path/in/bucket/file.txt", "local/path/to/save/downloaded_file.txt")

In [8]:
from google.cloud import bigquery
from google.oauth2 import service_account

class BigQueryTableManager:
    def __init__(self, project_id):
        self.project_id = project_id
        self.client = bigquery.Client(project=self.project_id)

    def create_table(self, dataset_id, table_id, schema):
        """Creates a new table in a BigQuery dataset."""
        dataset_ref = self.client.dataset(dataset_id)
        table_ref = dataset_ref.table(table_id)

        table = bigquery.Table(table_ref, schema=schema)
        try:
            table = self.client.create_table(table)  # API request
            print(f"Table {table.project}.{table.dataset_id}.{table.table_id} created.")
        except Exception as e:
            print(f"Error creating table: {e}")

    def update_table(self, dataset_id, table_id, rows_to_insert):
        """Inserts rows into an existing BigQuery table."""
        table_ref = self.client.dataset(dataset_id).table(table_id)

        try:
            errors = self.client.insert_rows_json(table_ref, rows_to_insert)  # API request
            if errors == []:
                print(f"Successfully inserted {len(rows_to_insert)} rows into {dataset_id}.{table_id}.")
            else:
                print(f"Encountered errors while inserting rows: {errors}")
        except Exception as e:
            print(f"Error updating table: {e}")

    def load_dataframe(self, dataset_id, table_id, dataframe):
        """Loads a pandas DataFrame into a BigQuery table."""
        table_ref = self.client.dataset(dataset_id).table(table_id)

        try:
            job = self.client.load_table_from_dataframe(dataframe, table_ref)  # API request
            job.result()  # Waits for the job to complete.
            print(f"Successfully loaded {len(dataframe)} rows into {dataset_id}.{table_id}.")
        except Exception as e:
            print(f"Error loading DataFrame into table: {e}")

    def run_query(self, query):
        """Runs a SQL query on BigQuery and returns the results."""
        try:
            query_job = self.client.query(query)  # API request
            results = query_job.result()  # Waits for the job to complete.
            print("Query executed successfully.")
            return results
        except Exception as e:
            print(f"Error running query: {e}")
            return None

# Example Usage:
# project_id = 'your-gcp-project-id'
# dataset_id = 'your-dataset-id'
# table_id = 'your-table-id'

# schema = [
#     bigquery.SchemaField("name", "STRING", mode="NULLABLE"),
#     bigquery.SchemaField("age", "INTEGER", mode="REQUIRED"),
# ]

# manager = BigQueryTableManager(project_id)

# To create a table:
# manager.create_table(dataset_id, table_id, schema)

# To insert rows:
# rows_to_insert = [
#     {"name": "John", "age": 30},
#     {"name": "Anna", "age": 25},
# ]
# manager.update_table(dataset_id, table_id, rows_to_insert)

# To load a DataFrame:
# import pandas as pd
# data = {'name': ['John', 'Anna'], 'age': [30, 25]}
# df_example = pd.DataFrame(data)
# manager.load_dataframe(dataset_id, table_id, df_example)

# To run a query:
# query = "SELECT * FROM `your-gcp-project-id.your-dataset-id.your-table-id` LIMIT 10"
# query_results = manager.run_query(query)
# if query_results:
#     for row in query_results:
#         print(row)

In [ ]:
project_id = 'micolegio-backup'
dataset_id = 'nubox_silver_openaq'
dataset_view_id = 'nubox_gold_openaq'
table_id = 'locations'
view_id = 'locations'

query = f"""
CREATE OR REPLACE VIEW `{project_id}.{dataset_view_id}.{view_id}` (
  ID OPTIONS(description="ID"),
  NAME OPTIONS(description="NAME"),
  LOCALITY OPTIONS(description="LOCALITY"),
  TIMEZONE OPTIONS(description="TIMEZONE"),
  COUNTRY OPTIONS(description="COUNTRY"),
  INSTRUMENTS OPTIONS(description="INSTRUMENTS"),
  SENSORS OPTIONS(description="SENSORS")
) AS SELECT
  tb.ID,
  tb.NAME,
  tb.LOCALITY,
  tb.TIMEZONE,
  tb.COUNTRY,
  tb.INSTRUMENTS,
  tb.SENSORS
FROM
  `{project_id}.{dataset_id}.{table_id}` AS tb
"""
query_results = manager.run_query(query)
if query_results:
    print(query_results)

Query executed successfully.


In [ ]:
project_id = 'micolegio-backup'
dataset_id = 'nubox_silver_openaq'
dataset_view_id = 'nubox_gold_openaq'
table_id = 'locations'
view_id = 'stations'

query = f"""
CREATE OR REPLACE VIEW `{project_id}.{dataset_view_id}.{view_id}` (
  ID OPTIONS(description="ID"),
  NAME OPTIONS(description="NAME"),
  LOCALITY OPTIONS(description="LOCALITY"),
  TIMEZONE OPTIONS(description="TIMEZONE"),
  COUNTRY OPTIONS(description="COUNTRY"),
  PROVIDER OPTIONS(description="PROVIDER"),
  COORDINATES OPTIONS(description="COORDINATES")
) AS SELECT
  tb.ID,
  tb.NAME,
  tb.LOCALITY,
  tb.TIMEZONE,
  tb.COUNTRY,
  tb.PROVIDER,
  tb.COORDINATES
FROM
  `{project_id}.{dataset_id}.{table_id}` AS tb
"""
query_results = manager.run_query(query)
if query_results:
    print(query_results)

Query executed successfully.


In [199]:
from time import sleep
import pandas as pd
import json, os

def main():
  api_key = init_ev()
  bucket_name = "nubox_bronze"
  project_id = 'micolegio-backup'
  dataset_id = 'nubox_silver_openaq'
  batch_date = ""
  open_aq = OpenAQ(api_key)


  ######## countries

  countries = str
  countries = open_aq.get_countries()

  local_countries_json_file = JsonFileAQ('countries', countries)
  response, batch_date, local_countries_json_file_name  = local_countries_json_file.create_local_json_file()
  print(f"Local file '{local_countries_json_file_name}' created successfully.")
  # Use the uploader to upload the file
  uploader = GoogleStorageUploader(bucket_name)
  destination_blob_name = f"raw_data_openaq/countries/{batch_date}/{local_countries_json_file_name}"
  uploader.upload_file(local_countries_json_file_name, destination_blob_name)

  # Clean up the local file
  os.remove(local_countries_json_file_name)
  print(f"'{local_countries_json_file_name}' removed successfully.")

  df = local_countries_json_file.get_normalized_dataframe(bucket_name)
  print(df.info())


  table_id = 'countries'
  manager = BigQueryTableManager(project_id)
  manager.load_dataframe(dataset_id, table_id, df)

  ############# locality

  locality = str
  locality = open_aq.get_localities()
  locality_json_file = JsonFileAQ('locality',locality)

  local_locality_json_file = JsonFileAQ('locality', locality)
  response, batch_date, local_locality_json_file_name  = local_locality_json_file.create_local_json_file()
  print(f"Local file '{local_locality_json_file_name}' created successfully.")
  # Use the uploader to upload the file
  uploader = GoogleStorageUploader(bucket_name)
  destination_blob_name = f"raw_data_openaq/locality/{batch_date}/{local_locality_json_file_name}"
  uploader.upload_file(local_locality_json_file_name, destination_blob_name)

  # Clean up the local file
  os.remove(local_locality_json_file_name)
  print(f"'{local_locality_json_file_name}' removed successfully.")

  df = local_locality_json_file.get_normalized_dataframe(bucket_name)
  print(df.info())


  table_id = 'locality'
  manager = BigQueryTableManager(project_id)
  manager.load_dataframe(dataset_id, table_id, df)


  ######## sensors


  view_id = 'sensors'
  table_id = 'locality'
  dataset_view_id = 'nubox_gold_openaq'
  query = f"""
    CREATE OR REPLACE VIEW `{project_id}.{dataset_view_id}.{view_id}` (
    SENSOR_ID,
    SENSOR_NAME
  ) AS
  SELECT DISTINCT
    SENSORS.id as sensor_id,
    SENSORS.name as sensor_name
  FROM `{project_id}.{dataset_id}.{table_id}`,
  unnest(`{project_id}.{dataset_id}.{table_id}`.sensors) as SENSORS
  """
  query_results = manager.run_query(query)

  query = f"""
    SELECT
    SENSOR_ID
    FROM `{project_id}.{dataset_view_id}.{view_id}`
  """
  query_results = manager.run_query(query)
  if query_results:
      # Convert query_results to a Python list
      sensor_ids_list = [row['SENSOR_ID'] for row in query_results]

  all_sensors = []
  for sensor_id in sensor_ids_list:
    sensor = open_aq.get_sensor(sensor_id)
    if len(sensor)>0:
      print(sensor)
      all_sensors.append({"id": sensor_id,"measurements" : sensor})
      sleep(5)
  print(all_sensors)
  local_sensor_json_file = JsonFileAQ('sensor', all_sensors)
  response, batch_date, local_sensor_json_file_name  = local_sensor_json_file.create_local_json_file()
  print(f"Local file '{local_sensor_json_file_name}' created successfully.")
  # Use the uploader to upload the file
  uploader = GoogleStorageUploader(bucket_name)
  destination_blob_name = f"raw_data_openaq/sensors/{batch_date}/{local_sensor_json_file_name}"
  uploader.upload_file(local_sensor_json_file_name, destination_blob_name)



__name__ = '__main__'
main()

Local file 'countries_2025-10-19.json' created successfully.
File countries_2025-10-19.json uploaded to raw_data_openaq/countries/2025-10-20/countries_2025-10-19.json in bucket nubox_bronze.
'countries_2025-10-19.json' removed successfully.
Blob raw_data_openaq/countries/2025-10-19/countries_2025-10-19.json downloaded to /content/countries_2025-10-19.json.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             100 non-null    int64 
 1   code           100 non-null    object
 2   name           100 non-null    object
 3   datetimeFirst  99 non-null     object
 4   datetimeLast   99 non-null     object
 5   parameters     99 non-null     object
dtypes: int64(1), object(5)
memory usage: 4.8+ KB
None
Successfully loaded 100 rows into nubox_silver_openaq.countries.
Local file 'locality_2025-10-19.json' created successfully.
File locality_

In [11]:
from datetime import datetime
import os, json
import pandas as pd

now = datetime.now()
batch_date = now.strftime("%Y-%m-%d")
bucket_name = "nubox_bronze"
project_id = 'micolegio-backup'
dataset_id = 'nubox_silver_openaq'
local_sensor_json_file_name = f"sensor_{batch_date}.json"
downloader = GoogleStorageUploader(bucket_name)
source_blob_name = f"raw_data_openaq/sensors/{batch_date}/{local_sensor_json_file_name}"
downloader.download_file(source_blob_name, local_sensor_json_file_name)



with open(local_sensor_json_file_name, 'r') as f:
    data = json.load(f)
df = pd.json_normalize(data,sep="_")

print(df.head(5))
# Clean up the local file
os.remove(local_sensor_json_file_name)
print(f"'{local_sensor_json_file_name}' removed successfully.")

table_id = 'air_quality_measurements.'
manager = BigQueryTableManager(project_id)
manager.load_dataframe(dataset_id, table_id, df)


Blob raw_data_openaq/sensors/2025-10-20/sensor_2025-10-20.json downloaded to sensor_2025-10-20.json.
     id                                       measurements
0  4418  [{'value': 35.3, 'flagInfo': {'hasFlags': Fals...
1  3510  [{'value': 27.0, 'flagInfo': {'hasFlags': Fals...
2  3511  [{'value': 26.9, 'flagInfo': {'hasFlags': Fals...
3  1738  [{'value': 23.5, 'flagInfo': {'hasFlags': Fals...
4   111  [{'value': 13.1, 'flagInfo': {'hasFlags': Fals...
'sensor_2025-10-20.json' removed successfully.
Error loading DataFrame into table: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/micolegio-backup/datasets/nubox_silver_openaq/tables/air_quality_measurements.?prettyPrint=false: Invalid table ID "air_quality_measurements.".
